# 🧠 Teach a Model to Read Emotions
### WLU_ML × AWS Club: Fine-tuning a Hugging Face model on Amazon SageMaker AI

In this notebook you'll take a model someone else already trained (**DistilBERT**) and teach it one new skill: reading the **emotion** in a sentence (sadness, joy, love, anger, fear or surprise).

You'll do it twice:

| Part | Where the training runs | Why |
|---|---|---|
| **Part 2** | Right here, inside this notebook | So you can see every step happen |
| **Part 3** | On a separate machine that SageMaker rents for you | This is how real teams train models |

---

### How to use this notebook
- A **code cell** has `[ ]` on its left. Click inside it and press **Shift + Enter** to run it.
- While a cell runs you'll see `[*]`. When it finishes, the `*` becomes a number.
- **Run cells in order, top to bottom**, and wait for each one to finish before starting the next.
- Each step has three short sections: **What we're doing**, **Why it matters** and **What you should see**. Read them! They're the part you'll remember.

> **Before you start:** check the top-right corner of the AWS Console. The region should be **US East (N. Virginia)**.

---
# Part 2: Fine-tune the model in this notebook

## Step 2.1: Install the libraries

**What we're doing:** installing four Python libraries:
- `transformers`: Hugging Face's library for loading and training models
- `datasets`: Hugging Face's library for downloading datasets
- `accelerate`: a helper that `transformers` uses behind the scenes to run training
- `sagemaker`: the AWS library we'll use in Part 3 to launch a cloud training job

**Why it matters:** Python itself knows nothing about machine learning. These libraries are the tools. They contain thousands of lines of tested code, so we don't have to write a training loop from scratch. The version ranges (like `<5`) pin the code to versions we tested, because libraries change their behaviour between major versions.

**What you should see:** after about a minute, the cell finishes. Red text about dependency *warnings* is fine. A line starting with `ERROR:` is not.

> ⚠️ **After this cell finishes, restart the kernel:** **Kernel → Restart Kernel… → Restart.**
> Python loads libraries when it starts up. Restarting makes it pick up the versions you just installed instead of older ones that were already loaded.

In [ ]:
%pip install -q "transformers>=4.44,<5" "datasets>=2.20" "accelerate>=0.26" "sagemaker>=2.200,<3"

## Step 2.2: Check that everything loaded

**What we're doing:** importing the libraries and printing their versions, plus whether a GPU is available.

**Why it matters:** this is a quick health check. If something is broken, it's much easier to catch it here than halfway through training. Engineers do this all the time before starting anything expensive.

We also check for a **GPU**. GPUs are chips that do thousands of small calculations at once, which makes training much faster. We're deliberately on a cheap **CPU** machine (`ml.t3.medium`) because it's free-tier eligible, and our model and dataset are small enough that a CPU can handle them.

**What you should see:** three version numbers and `GPU available? False`. **False is correct here.**

In [ ]:
import torch, transformers, datasets

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("torch:", torch.__version__)
print("GPU available?", torch.cuda.is_available())

## Step 2.3: Load the dataset from Hugging Face

**What we're doing:** downloading the `dair-ai/emotion` dataset from the Hugging Face Hub and looking at a few examples.

**Why it matters:** a model learns from **examples**. Each example in this dataset has two parts:
- **x**: the input (a short tweet)
- **y**: the correct answer (the emotion label)

Learning from examples where the right answer is already known is called **supervised learning**. It's like studying with an answer key.

Notice that the labels are stored as **numbers** (0–5), not words. Models only work with numbers, so `labels` is our "decoder ring" for turning a number back into a word.

**What you should see:** a summary showing `train` (16,000 rows), `validation` (2,000) and `test` (2,000), the 6 label names, and 5 example tweets with their emotions.

> 💡 Optional: open **huggingface.co/datasets/dair-ai/emotion** in a new tab to browse the dataset yourself.

In [ ]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion")
print(ds)

labels = ds["train"].features["label"].names
print("Labels:", labels)

print("\nA few examples:")
for i in range(5):
    row = ds["train"][i]
    print(f"  {labels[row['label']]:>9} | {row['text']}")

## Step 2.4: Take a small slice of the data

**What we're doing:** shuffling the data, then keeping **2,000 examples for training** and **500 for testing**.

**Why it matters:** there are two separate ideas here.

1. **Why only 2,000?** Training time grows with the amount of data. On a small CPU, 2,000 examples takes about 10 minutes, while 16,000 would take over an hour. Because the model is already pretrained, a small slice is still enough to teach it something useful.
2. **Why a separate test set?** If you only check a model on data it trained on, you can't tell whether it *learned the pattern* or just *memorized the answers*. The test set is the **final exam**: the model never sees these sentences during training.

**Why shuffle with `seed=42`?** Shuffling makes sure our slice is a random mix and not, say, the first 2,000 rows (which might all share a theme). The **seed** makes the shuffle repeatable, so everyone in the room gets the same slice and similar results.

**What you should see:** `Training examples: 2000` and `Test examples: 500`.

In [ ]:
train_ds = ds["train"].shuffle(seed=42).select(range(2000))
test_ds  = ds["test"].shuffle(seed=42).select(range(500))

print("Training examples:", len(train_ds))
print("Test examples:", len(test_ds))

## Step 2.5: Tokenization (how a model "reads")

**What we're doing:** loading DistilBERT's **tokenizer** and using it on one sentence.

**Why it matters:** neural networks can only do math on numbers. They can't read letters. A **tokenizer** bridges that gap in three moves:

1. **Split** the text into pieces called *tokens* (usually words, sometimes parts of words).
2. **Add special markers**: `[CLS]` at the start (the model uses this spot to summarize the whole sentence) and `[SEP]` at the end.
3. **Look up** each token's ID number in a fixed vocabulary (DistilBERT's has 30,522 entries).

Every model has its own matching tokenizer. If you used a different tokenizer, the IDs wouldn't mean what the model expects, so we load it by the same name as the model.

**What you should see:** a list of numbers, then the matching tokens: `['[CLS]', 'i', 'feel', 'so', 'happy', 'today', '[SEP]']`.

> 🧪 **Try it:** change `example` to a long or unusual word like `"unbelievably"` and rerun. Rare words get split into smaller pieces marked with `##`. That's how the model handles words it has never seen whole.

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

example = "i feel so happy today"
encoded = tokenizer(example)
print("Token IDs:", encoded["input_ids"])
print("Tokens:   ", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

## Step 2.6: Tokenize the whole dataset

**What we're doing:** running the tokenizer over all 2,000 training and 500 test sentences.

**Why it matters:** the model trains on **batches** (groups of sentences processed together), and a batch has to be a neat rectangle of numbers. Sentences have different lengths, so we make them all the same length:
- `padding="max_length"`: short sentences get filled with `0`s (padding) up to 64 tokens.
- `truncation=True`: anything longer than 64 tokens gets cut off.
- `max_length=64`: tweets are short, so 64 is plenty.

The tokenizer also creates an **attention mask** (1 = real token, 0 = padding) so the model knows to ignore the filler.

`batched=True` processes many sentences at once instead of one at a time, which is much faster.

**What you should see:** two progress bars, then `Done tokenizing!`.

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_tok = train_ds.map(tokenize, batched=True)
test_tok  = test_ds.map(tokenize, batched=True)
print("Done tokenizing!")

## Step 2.7: Load the pretrained model

**What we're doing:** downloading DistilBERT and attaching a new **classifier head** with 6 outputs, one per emotion.

**Why it matters:** this is the heart of **transfer learning**.
- The main body of DistilBERT was already trained on a huge amount of English text. It "knows" grammar, context and word meanings. That knowledge lives in about **67 million numbers** called **weights**.
- On top, we add a small new layer (the **classifier head**) that turns the model's understanding of a sentence into 6 scores. This head starts out **random**: it has never seen our task.

We also give the model `id2label` and `label2id`, so its predictions come back as readable names like `"joy"` instead of just `1`.

**What you should see:** `Model loaded with ~67,000,000 parameters`. You'll **also** see a warning like:

> *Some weights of DistilBertForSequenceClassification were not initialized… You should probably TRAIN this model…*

**That warning is expected and good news.** It's telling you exactly what we just described: the new classifier head is random and needs training.

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
print(f"Model loaded with {model.num_parameters():,} parameters")

## Step 2.8: Test the model BEFORE training

**What we're doing:** writing a small `predict()` function and asking the untrained model about four sentences.

**Why it matters:** you can't tell how much training helped unless you know where you started. This is your **baseline**.

Here's what happens inside `predict()`:
1. **Tokenize** the sentence (same as Step 2.5).
2. Run it through the model (the **forward pass**). The model outputs 6 raw scores called **logits**.
3. **Softmax** turns those scores into probabilities that add up to 100%:
   $$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$
4. Pick the emotion with the highest probability.

Two lines are worth knowing:
- `model.eval()` switches the model into "prediction mode" (it turns off some randomness used only during training).
- `torch.no_grad()` tells PyTorch not to keep track of the math needed for learning. We're only predicting, so this saves memory and time.

**What you should see:** basically random guesses with low confidence, around 16–25% each. With 6 emotions, random guessing is 1 in 6 ≈ 17%. **Remember these answers. We'll compare after training.**

In [ ]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    top = int(probs.argmax())
    return f"{id2label[top]} ({probs[top]:.0%} sure)"

test_sentences = [
    "i just got accepted into my dream internship",
    "i miss my family so much it hurts",
    "why would you lie to me like that",
    "i heard a noise downstairs and i am home alone",
]

print("BEFORE training:")
for s in test_sentences:
    print(f"  {s!r} -> {predict(s)}")

## Step 2.9: Train! (about 8–12 minutes)

**What we're doing:** setting the training options and starting training with Hugging Face's `Trainer`.

**Why it matters:** this is where the model actually learns. Each **training step** follows the same loop:

| Step | What happens | In math |
|---|---|---|
| 1. **Predict** | Run a batch of 16 sentences through the model | forward pass |
| 2. **Measure** | Compare predictions to the right answers. The **loss** is a single number for "how wrong" | $L = -\log p_{\text{true}}$ |
| 3. **Blame** | **Backpropagation** works out how much each weight contributed to the error | $\partial L / \partial w$ |
| 4. **Nudge** | Move every weight a tiny step in the direction that lowers the loss | $w \leftarrow w - \eta \cdot \partial L / \partial w$ |

What each setting means:
- `num_train_epochs=2`: an **epoch** is one full pass over all 2,000 examples. We do 2.
- `per_device_train_batch_size=16`: look at 16 examples, then update. 2,000 ÷ 16 = 125 steps per epoch, × 2 epochs = **250 steps**.
- `learning_rate=5e-5`: the size of each nudge (η). Too big and the model overshoots; too small and it barely learns.
- `eval_strategy="epoch"`: check accuracy on the test set after each epoch.
- `save_strategy="no"` and `report_to="none"`: skip saving checkpoints and sending logs to outside tools, to keep things simple.

`compute_metrics` tells the Trainer how to score the model: **accuracy** = the fraction of predictions that are correct.

**What you should see:** a progress bar counting up to **250 steps**, and a table where **Training Loss goes down** and **Accuracy** appears after each epoch.

> 🐢 **Too slow?** If the estimate is over 15 minutes, stop the cell (■ button), change `range(2000)` to `range(1000)` in Step 2.4, then rerun Steps 2.4 → 2.9.

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == true_labels).mean())}

training_args = TrainingArguments(
    output_dir="emotion-model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

## Step 2.10: Test the model AFTER training

**What we're doing:** measuring accuracy on the 500 test sentences, then asking the same four questions as Step 2.8.

**Why it matters:** this is the payoff, and a fair test.
- The **accuracy** number comes from sentences the model **never trained on**, so it tells us whether the model learned general patterns.
- Comparing the four sentences with Step 2.8 shows the change directly: same model, same inputs, but the weights are different now.

**What you should see:** accuracy well above 17% (usually about 60–85% with this small slice), and sensible labels for the four sentences, with much higher confidence than before.

> 🤔 **Think about it:** 2,000 examples and ~10 minutes took this model from random guessing to genuinely useful. It could only learn that fast because DistilBERT already understood English. That's the power of starting from a pretrained model.

In [ ]:
results = trainer.evaluate()
print(f"Accuracy on unseen test data: {results['eval_accuracy']:.1%}\n")

print("AFTER training:")
for s in test_sentences:
    print(f"  {s!r} -> {predict(s)}")

## Step 2.11: Try your own sentences

**What we're doing:** playing with your model.

**Why it matters:** poking at a model is how you find its limits. Try to **trick it**:
- **Sarcasm:** "oh great, another monday" (it usually gets this wrong)
- **Mixed feelings:** "i'm excited but terrified for my first day"
- **Negation:** "i am not happy at all"

When it fails, ask yourself *why*. Usually the answer is in the **training data**: if the dataset has very few sarcastic tweets, the model never got a chance to learn sarcasm. **A model can only learn the patterns it was shown.**

In [ ]:
my_sentence = "type anything you want here"
print(predict(my_sentence))

---
# Part 3: Do it the cloud way, with a SageMaker training job

So far, everything ran **inside this notebook**, on a small machine. That's great for experimenting, but real teams don't train important models this way. Instead they:

1. Put the training code into a **script** (`train.py`).
2. Ask **SageMaker** to rent a fresh machine of whatever size they need (including big GPUs).
3. SageMaker runs the script inside a **Docker container** (a packaged, identical environment).
4. The trained model is saved to **S3** (AWS's file storage).
5. **The machine shuts itself down**, so you only pay for the minutes it ran.

Your notebook stays small and cheap. It just sends instructions.

## Step 3.1: Make a folder for the training code

**What we're doing:** creating a folder called `code/`.

**Why it matters:** SageMaker uploads **a whole folder** to the training machine, not just one file. Keeping the script and its requirements together in one folder means the remote machine gets everything it needs and nothing it doesn't.

In [ ]:
import os
os.makedirs("code", exist_ok=True)
print("Folder created")

## Step 3.2: List the libraries the training machine needs

**What we're doing:** writing a `requirements.txt` file. The first line, `%%writefile`, is a notebook trick: it **saves the cell's contents to a file** instead of running them.

**Why it matters:** the training machine is a **brand-new computer**. It doesn't have the libraries we installed in Step 2.1. SageMaker reads `requirements.txt` and installs everything listed before running our script.

Notice that we pin **exact** versions here (`==`). On a remote machine we can't watch, we want zero surprises: the same code with the same libraries gives the same result every time. This idea is called **reproducibility**.

**What you should see:** `Writing code/requirements.txt`

In [ ]:
%%writefile code/requirements.txt
transformers==4.44.2
datasets==2.21.0
accelerate==0.33.0

## Step 3.3: Write the training script

**What we're doing:** saving all of Part 2's logic into one file, `code/train.py`.

**Why it matters:** a notebook is interactive, but a **script** can run start to finish with no human clicking anything. That's what a remote machine needs. Read through it: you'll recognize every piece from Part 2.

Two new things to notice:
- **`argparse`** lets us pass settings (like `--epochs 2`) from outside the script, so we can change them without editing the code. SageMaker passes our **hyperparameters** this way.
- **`SM_MODEL_DIR`** is a folder path that SageMaker gives the script. **Anything saved there is automatically uploaded to S3 when the job finishes.** That's how the trained model gets back to us.

**What you should see:** `Writing code/train.py`. In the left file browser, open the `code` folder to see both files.

In [ ]:
%%writefile code/train.py
import argparse
import os

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--train_size", type=int, default=2000)
    args = parser.parse_args()

    # SageMaker tells us where to save the model via this environment variable.
    # Anything saved here gets uploaded to S3 automatically when the job ends.
    model_dir = os.environ.get("SM_MODEL_DIR", "model")
    model_name = "distilbert-base-uncased"

    ds = load_dataset("dair-ai/emotion")
    labels = ds["train"].features["label"].names
    train_ds = ds["train"].shuffle(seed=42).select(range(args.train_size))
    test_ds = ds["test"].shuffle(seed=42).select(range(500))

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

    train_tok = train_ds.map(tokenize, batched=True)
    test_tok = test_ds.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label={i: n for i, n in enumerate(labels)},
        label2id={n: i for i, n in enumerate(labels)},
    )

    def compute_metrics(eval_pred):
        logits, true_labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": float((preds == true_labels).mean())}

    training_args = TrainingArguments(
        output_dir="/tmp/checkpoints",
        num_train_epochs=args.epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=5e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=25,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    print(f"FINAL TEST ACCURACY: {results['eval_accuracy']:.1%}")

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"Model saved to {model_dir}")


if __name__ == "__main__":
    main()

## Step 3.4: Connect to SageMaker and check the container exists

**What we're doing:** getting three things SageMaker needs, then checking that AWS has the right training container.

**Why it matters:** each of these is a core AWS idea:
- **Role** (IAM): a **permission slip**. AWS doesn't let services touch your resources unless you say so. The role lets SageMaker read and write your S3 bucket on your behalf.
- **Default bucket** (S3): where the trained model will land. SageMaker creates it for you, named after your region and account.
- **Training container** (Docker): a pre-built environment that AWS maintains, with Python and PyTorch already installed. Our script runs **inside** it. This is exactly what Docker is for: the same environment on any machine, every time.

The `image_uris.retrieve(...)` call is a **safety check**. It asks AWS whether a CPU training container exists for PyTorch 2.1 in your region. If it doesn't, you'll get an error here, in a few seconds, rather than after waiting 5 minutes for a job to fail.

**What you should see:**
- `Region: us-east-1`
- a role like `arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-…`
- a bucket like `sagemaker-us-east-1-123456789012`
- a container address ending in something like `pytorch-training:2.1-cpu-py310`

In [ ]:
import sagemaker
from sagemaker import image_uris

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name

print("Region:", region)
print("Role:", role)
print("Default S3 bucket:", session.default_bucket())

# Sanity check: make sure AWS has a CPU PyTorch training container for this version
image = image_uris.retrieve(
    framework="pytorch", region=region, version="2.1", py_version="py310",
    instance_type="ml.m5.xlarge", image_scope="training",
)
print("Training container:", image)

## Step 3.5: Launch the training job

**What we're doing:** describing the job (which script, which container, which machine) and starting it with `.fit()`.

**Why it matters:** this one cell does what would otherwise take an hour of manual setup: rent a machine, install Docker, copy your code over, install libraries, run training, upload results and shut everything down. Reading the settings:
- `entry_point` / `source_dir`: run `train.py` from the `code/` folder
- `framework_version` / `py_version`: which PyTorch container to use
- `instance_type="ml.m5.xlarge"`: the machine to rent (4 CPUs, free-tier eligible for training)
- `hyperparameters`: passed to our script as `--epochs 2 --train_size 2000`

> **Why `PyTorch` and not SageMaker's `HuggingFace` estimator?** The Hugging Face training containers only support **GPU** machines. Using the PyTorch container plus our `requirements.txt` keeps us on a free-tier CPU. On an account with GPU access, the `HuggingFace` estimator is the usual choice.

**What you should see:** logs streaming in, in this order:
1. `Starting - Starting the training job…`
2. `Starting - Preparing the instances for training…` ⏳ (**3–5 minutes**: AWS is starting a machine just for you)
3. `Downloading - Downloading the training image…`
4. `Training - …` then `pip install` output, then the same progress bars as Part 2
5. `FINAL TEST ACCURACY: …%`
6. `Completed - Training job completed`, plus **Billable seconds**

> 🔎 **While you wait:** open the AWS Console tab → **SageMaker AI** → **Training** → **Training jobs** → click `emotion-workshop-…`. You can watch the status, the machine type, the logs and where the model will be saved.

In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",          # the script to run
    source_dir="code",               # folder uploaded to the job (includes requirements.txt)
    role=role,
    framework_version="2.1",
    py_version="py310",
    instance_type="ml.m5.xlarge",    # free-tier eligible CPU machine
    instance_count=1,
    hyperparameters={"epochs": 2, "train_size": 2000},
    base_job_name="emotion-workshop",
)

estimator.fit(wait=True)

## Step 3.6: Find your trained model in S3

**What we're doing:** printing the S3 address where SageMaker saved the model.

**Why it matters:** the training machine is **gone**. It shut down when the job finished. The only thing left is what it saved to S3. That's a key cloud idea: **compute is temporary, storage is permanent**. You rent computers briefly and keep the results in storage.

The model is packed into `model.tar.gz`, a compressed folder (like a `.zip`) holding the trained weights and the tokenizer.

**What you should see:** a path like `s3://sagemaker-us-east-1-…/emotion-workshop-…/output/model.tar.gz`

> 🖱️ To see it in the console: search **S3** → open the `sagemaker-us-east-1-…` bucket → `emotion-workshop-…` → `output`.

In [ ]:
print("Your trained model is here:")
print(estimator.model_data)

## Step 3.7 (optional): Use the model the cloud trained

**What we're doing:** downloading `model.tar.gz` from S3, unpacking it, and loading it into a Hugging Face `pipeline`.

**Why it matters:** this proves the whole round trip worked. A model trained on a **completely different machine** can be loaded anywhere: your notebook, a website, an app. Saving and reloading models like this is how they get **deployed**.

A `pipeline` bundles the tokenizer, the model and softmax into one easy function. It does the same work as our `predict()` function from Step 2.8.

**What you should see:** predictions similar to Step 2.10.

In [ ]:
!aws s3 cp {estimator.model_data} ./job-model.tar.gz
!mkdir -p job-model && tar -xzf job-model.tar.gz -C job-model

from transformers import pipeline
cloud_clf = pipeline("text-classification", model="job-model")

for s in test_sentences:
    print(f"  {s!r} -> {cloud_clf(s)[0]}")

---
# Part 4: Clean up (do not skip!)

**Why it matters:** in the cloud, **you pay for what's running**, not what you're using. The training job already shut itself down, but **this notebook's machine keeps running** (and counting toward your free-tier hours, then your bill) until you stop it.

1. In JupyterLab: **File → Shut Down** → confirm.
2. In the **SageMaker Studio** tab: left panel → **JupyterLab** → find `emotion-workshop` → click **Stop**. Wait until it says **Stopped**.
3. Double-check: Studio's left panel → **Running instances** should show **nothing**.

You don't need to delete the training job (finished jobs cost nothing). The model file in S3 is about 250 MB, within the S3 free tier. If you want it gone: **S3** → your `sagemaker-us-east-1-…` bucket → select the `emotion-workshop-…` folder → **Delete**.

---

## 🎉 What you just did

- Downloaded a **pretrained model** and a **dataset** from Hugging Face
- Turned text into numbers with a **tokenizer**
- **Fine-tuned** the model and measured it on a **test set** it had never seen
- Ran the same training as a **SageMaker training job**: a rented machine, a **Docker** container, an **IAM role** and **S3** storage

That's the same shape as real-world ML, just at a smaller scale. Thanks for coming! — **WLU_ML × AWS Club**